# RF-DETR GPU gate on Google Colab

Runs `scripts/run_gate.sh` (the executable part of the **gpu-verify** checklist in `AGENTS.md`) on whatever GPU this runtime has, so the TensorRT + CUDA/DALI paths get checked on cards we do not own: T4 (sm_75), L4 (sm_89), A100 (sm_80), G4 / RTX PRO 6000 Blackwell (sm_120).

**Before you start:** *Runtime → Change runtime type* → pick the GPU, and turn **High-RAM on** where offered — the standard runtime has 2 vCPUs and ~12 GB, which makes the nvcc + TensorRT builds slow and can OOM a parallel build.

What a Colab run proves and does not: see `specs/rented-gpu-runbook.md` → *Google Colab*. `UNRUN` lines in the summary are unrun, not passed.

In [ ]:
# --- Configuration ------------------------------------------------------------
BRANCH = "develop"                       # branch or tag to verify
REPO_URL = "https://github.com/olibartfast/rf-detr-cpp-inference.git"
# A segmentation .onnx exported at 432 (runbook Part 1). If it is not on Drive,
# cell 4 exports one here with the pinned rfdetr.
MODEL_ON_DRIVE = "/content/drive/MyDrive/rfdetr-gate/rfdetr-seg-medium.onnx"
VIDEO_FRAMES = 1000                      # checklist wants 1000; 300 keeps compute-sanitizer short
SKIP_DALI = False                        # True: skip DALI staging and its builds
SKIP_BUILD_MATRIX = True                 # independent-halves + guards are card-independent
SKIP_DEFAULT_PATH = True                 # the CPU-only ONNX Runtime build needs no GPU

In [ ]:
# 1. Drive holds the results, so a recycled runtime does not take them with it.
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 2. Clone.
%cd /content   # a re-run must not start inside the clone it deletes
!rm -rf /content/rfdetr_inference
!git clone --depth 1 --branch {BRANCH} {REPO_URL} /content/rfdetr_inference
%cd /content/rfdetr_inference
!git log --oneline -1

In [ ]:
# 3. Toolchain: pinned CUDA toolkit (+ forward compat if the driver is older),
#    FFmpeg/SDL2 dev packages, DALI from pip. ~5-10 min.
import os
os.environ['SKIP_DALI'] = '1' if SKIP_DALI else '0'
!./scripts/colab/setup_colab.sh 2>&1 | tail -40

In [ ]:
# 4. Model: from Drive if present, otherwise export a 432 segmentation model.
import glob, os, shutil
MODEL = '/content/model.onnx'
if os.path.exists(MODEL_ON_DRIVE):
    shutil.copy(MODEL_ON_DRIVE, MODEL)
else:
    !pip install -q -r deploy/requirements.txt
    !python deploy/export_segmentation.py --model_type medium --input_size 432
    exported = sorted(glob.glob('output/*.onnx'))
    assert exported, 'export produced no .onnx — see the output above'
    shutil.copy(exported[0], MODEL)
    os.makedirs(os.path.dirname(MODEL_ON_DRIVE), exist_ok=True)
    shutil.copy(exported[0], MODEL_ON_DRIVE)   # reuse it on the next GPU
print(MODEL, os.path.getsize(MODEL), 'bytes')

In [ ]:
# 5. Video of exactly VIDEO_FRAMES frames for compute-sanitizer.
VIDEO = '/content/long.mp4'
!ffmpeg -y -loglevel error -loop 1 -i data/dog.jpg -frames:v {VIDEO_FRAMES} -r 25 -pix_fmt yuv420p {VIDEO}
!ffprobe -v error -count_frames -select_streams v:0 -show_entries stream=nb_read_frames -of csv=p=0 {VIDEO}

In [ ]:
# 6. Launch the gate in the background; results go to Drive, one directory per card.
import datetime, subprocess
gpu = subprocess.check_output(['nvidia-smi', '--query-gpu=name', '--format=csv,noheader'], text=True)
gpu = gpu.splitlines()[0].strip().replace(' ', '_')
RESULTS = f"/content/drive/MyDrive/rfdetr-gate/{gpu}-{datetime.datetime.now():%Y%m%d-%H%M}"
os.makedirs(RESULTS, exist_ok=True)
env = (f"source /content/rfdetr_gate_env.sh && "
       f"MODEL={MODEL} VIDEO={VIDEO} RESULTS={RESULTS} "
       f"SKIP_BUILD_MATRIX={int(SKIP_BUILD_MATRIX)} SKIP_DEFAULT_PATH={int(SKIP_DEFAULT_PATH)}")
gate = subprocess.Popen(['bash', '-c', f"{env} ./scripts/run_gate.sh > {RESULTS}/run.log 2>&1"])
print('gate pid', gate.pid, '->', RESULTS)

In [ ]:
# 7. Follow it. Re-run this cell after a browser reconnect; the gate keeps going
#    as long as the runtime is alive.
import time
from IPython.display import clear_output
while gate.poll() is None:
    clear_output(wait=True)
    !tail -n 15 {RESULTS}/gate.log
    time.sleep(30)
print('gate exited with', gate.returncode)

In [ ]:
# 8. Summary — copy this and environment.txt into CHANGELOG.md (gate step 7).
!cat {RESULTS}/gate.summary
!cat {RESULTS}/environment.txt
!ls -la {RESULTS}

In [ ]:
# 9. --display on a headless runtime: Xvfb window of the video's size, real frame
#    content, and 'q' ending the run early. Plain, then the full GPU pipeline.
#    Needs the build from the gate (step 1).
!APP=build-gpu/inference_app MODEL={MODEL} VIDEO={VIDEO} OUT={RESULTS}/display-cpu \
    ./scripts/check_display.sh 2>&1 | tee {RESULTS}/display-cpu.txt
!APP=build-gpu/inference_app MODEL={MODEL} VIDEO={VIDEO} OUT={RESULTS}/display-gpu \
    ./scripts/check_display.sh --gpu-preprocess --gpu-postprocess --segmentation \
    2>&1 | tee {RESULTS}/display-gpu.txt

In [ ]:
# 10. Look at what the window showed.
from IPython.display import Image, display
for d in ('display-cpu', 'display-gpu'):
    display(Image(f'{RESULTS}/{d}/window.png', width=480))